# Can FHE Protect AI Agents?
## Interactive Demo

This notebook walks through the core contribution of our paper: **using Fully Homomorphic Encryption (FHE) and Private Information Retrieval (PIR) to protect AI agent tool-calls from honest-but-curious servers.**

**The scenario:** A healthcare AI agent needs to:
1. **Look up drug interactions** in an external database (PIR)
2. **Compute a patient risk score** on an external scoring server (FHE)
3. **Make a treatment decision** locally

Without encryption, the database server sees *which drugs* the agent queries, and the scoring server sees the *patient's medical features*. With our approach, servers process **encrypted data** and learn **nothing** about the query content.

**What you'll see:**
- Plaintext baseline (full information exposure)
- PIR-encrypted drug lookup (server learns nothing)
- FHE-encrypted risk scoring (server computes on ciphertext)
- Full end-to-end pipeline with timing and correctness verification

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────
import sys, os, json, time
from pathlib import Path

# Ensure project root is on sys.path
PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "src").exists():
    pass  # already at project root
elif (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

# Core project imports
from src.utils.data_utils import load_drug_db, load_patients
from src.prototype.encrypted_lookup import (
    EncryptedDrugLookup, DrugRecord, load_drug_records_from_json,
)
from src.prototype.encrypted_scoring import (
    EncryptedScoringTool, PatientFeatures, cardiovascular_risk_model,
)
from src.feasibility.fhe_computation import FHEParams

print(f"Project root: {PROJECT_ROOT}")
print(f"NumPy {np.__version__}, Pandas {pd.__version__}")

try:
    import tenseal as ts
    print(f"TenSEAL {ts.__version__} (CKKS/BFV backend)")
except ImportError:
    print("WARNING: TenSEAL not installed — encrypted cells will fail")

try:
    import matplotlib
    import matplotlib.pyplot as plt
    %matplotlib inline
    plt.rcParams.update({"figure.dpi": 120, "font.size": 11})
    print(f"Matplotlib {matplotlib.__version__}")
except ImportError:
    print("WARNING: matplotlib not installed — plot cells will fail")

print("\nReady.")

## Cell 2 — Load Data

Our prototype uses two datasets (both synthetic, deterministic, seed=42):

| Dataset | Records | Purpose |
|---------|---------|---------|
| Drug interactions | 10,000 pairs | PIR lookup database |
| Patient profiles | 1,000 patients | FHE risk scoring inputs |

Each drug record contains: drug pair names, severity level (none/mild/moderate/severe/contraindicated), mechanism, and clinical recommendation. Each patient has 8 clinical features: age, BMI, blood pressure (systolic/diastolic), cholesterol, glucose, medications count, and conditions count.

In [ ]:
# ── Load datasets ──────────────────────────────────────────────────────
# Use the 200-record subset for fast demo; the full 10K DB is used in benchmarks.
DB_SIZE = "200"

try:
    drug_raw = load_drug_db(DB_SIZE)
    patients_raw = load_patients("10")
except FileNotFoundError:
    print("Datasets not found. Generating them now...")
    from src.utils.data_utils import generate_all_datasets
    generate_all_datasets(seed=42)
    drug_raw = load_drug_db(DB_SIZE)
    patients_raw = load_patients("10")

print(f"Loaded {len(drug_raw):,} drug interaction records")
print(f"Loaded {len(patients_raw)} patient profiles\n")

# ── Display sample drug records ───────────────────────────────────────
drug_df = pd.DataFrame(drug_raw)
display_cols = ["drug_a", "drug_b", "severity", "mechanism"]
print("Sample drug interactions:")
drug_df[display_cols].head(8)

In [ ]:
# ── Display sample patient profiles ────────────────────────────────────
pat_df = pd.DataFrame(patients_raw)
pat_cols = ["patient_id", "age", "bmi", "systolic_bp", "diastolic_bp",
            "cholesterol_total", "glucose_fasting", "num_medications"]
print("Sample patient profiles:")
pat_df[pat_cols].head(6)

## Cell 3 — The Problem: Plaintext Tool Calls

When an AI agent calls a tool server in plaintext, the server sees **everything**: which drugs the agent looked up, and all of the patient's medical features. This is the baseline we want to protect against.

Below, we simulate what an honest-but-curious server would log during a plaintext interaction. Notice the server captures the exact drug pair and severity — this is **full content leakage**.

In [ ]:
# ── The Problem: plaintext tool calls ─────────────────────────────────
# Load the drug database into the lookup tool
drug_records = load_drug_records_from_json(
    PROJECT_ROOT / f"data/drug_interactions_{DB_SIZE}.json"
)
lookup = EncryptedDrugLookup()
lookup.load_drug_database(drug_records)

# Pick a real drug pair from the database
pairs = lookup.drug_pairs
drug_a, drug_b = pairs[42]  # deterministic choice

print("=" * 60)
print("  PLAINTEXT TOOL CALL (no encryption)")
print("=" * 60)

# --- Plaintext drug lookup (server sees everything) ---
print(f"\nAgent sends: query(drug_a='{drug_a}', drug_b='{drug_b}')")
record = lookup.query_plaintext(drug_a, drug_b)
print(f"\n  SERVER LOG: Query for drug_a='{drug_a}', drug_b='{drug_b}'")
print(f"  SERVER LOG: Returned severity='{record.severity}', "
      f"desc='{record.description[:60]}...'")
print(f"  SERVER LOG: >>> FULL CONTENT VISIBLE <<<")

# --- Plaintext risk scoring (server sees all features) ---
p = patients_raw[0]
features = PatientFeatures(
    age=p["age"], bmi=p["bmi"],
    blood_pressure_systolic=p["systolic_bp"],
    blood_pressure_diastolic=p["diastolic_bp"],
    cholesterol=p["cholesterol_total"],
    glucose=p["glucose_fasting"],
    num_medications=p["num_medications"],
    num_conditions=p["num_conditions"],
)
print(f"\nAgent sends patient features to scoring server:")
print(f"  SERVER LOG: age={features.age}, bmi={features.bmi:.1f}, "
      f"bp={features.blood_pressure_systolic}/{features.blood_pressure_diastolic}, "
      f"chol={features.cholesterol:.0f}, glucose={features.glucose:.0f}")
print(f"  SERVER LOG: >>> ALL MEDICAL DATA VISIBLE <<<")

# Compute plaintext score
weights, bias, activation = cardiovascular_risk_model()
scorer = EncryptedScoringTool(FHEParams.deep())
scorer.setup(weights, bias, activation_coeffs=activation)
plain_score = scorer.score_plaintext(features)
print(f"\n  Risk score (plaintext): {plain_score:.4f}")
print(f"\n  An honest-but-curious server can reconstruct the")
print(f"  patient's full medical profile and the agent's intent.")

## Cell 4 — PIR: Private Drug Lookup

**Private Information Retrieval (PIR)** lets the agent query a database without the server learning *which* record was requested.

**How it works:**
1. The agent knows the index of the drug pair it wants (e.g., index 42 out of 200 records).
2. The agent encrypts the index into a BFV ciphertext — a blob of bytes that encodes "give me record 42" without revealing "42."
3. The server multiplies every record in the database by the encrypted query (homomorphically). Only record 42 survives; the rest cancel out.
4. The agent decrypts the response to get the plaintext record.

The server processes the **entire database** on every query — it can't tell which record was selected. This is the computational cost of privacy.

In [ ]:
# ── PIR: Private Drug Lookup ───────────────────────────────────────────
print("=" * 60)
print("  PIR-ENCRYPTED DRUG LOOKUP")
print("=" * 60)

# Step 1: Setup PIR over the drug database
print("\n[1/4] Setting up PIR scheme...")
t0 = time.perf_counter()
lookup.setup_pir()
setup_ms = (time.perf_counter() - t0) * 1000
print(f"  PIR setup complete ({setup_ms:.0f} ms, {len(drug_records)} records)")

# Step 2: Agent creates encrypted query
print(f"\n[2/4] Agent creates encrypted query for ({drug_a}, {drug_b})...")
t0 = time.perf_counter()
encrypted_query = lookup.create_query(drug_a, drug_b)
query_ms = (time.perf_counter() - t0) * 1000
print(f"  Query generated ({query_ms:.0f} ms)")
print(f"  Encrypted query size: {len(encrypted_query):,} bytes")
print(f"  First 80 bytes (hex): {encrypted_query[:80].hex()}")
print(f"  >>> This is what the server receives. It's indistinguishable")
print(f"      from a query for ANY other drug pair. <<<")

# Step 3: Server processes encrypted query
print(f"\n[3/4] Server processes encrypted query...")
t0 = time.perf_counter()
encrypted_result = lookup.server_process(encrypted_query)
server_ms = (time.perf_counter() - t0) * 1000
n_parts = len(encrypted_result)
total_resp = sum(len(p) for p in encrypted_result)
print(f"  Server computation: {server_ms:.0f} ms")
print(f"  Encrypted response: {n_parts} parts, {total_resp:,} bytes total")
print(f"  SERVER LOG: Processed encrypted query. Learned: NOTHING.")

# Step 4: Agent decrypts result
print(f"\n[4/4] Agent decrypts result...")
t0 = time.perf_counter()
result = lookup.decrypt_result(encrypted_result)
decrypt_ms = (time.perf_counter() - t0) * 1000
total_ms = query_ms + server_ms + decrypt_ms

print(f"  Decrypted in {decrypt_ms:.0f} ms")
print(f"\n  Result: {result.drug_a} + {result.drug_b}")
print(f"  Severity: {result.severity}")
print(f"  Description: {result.description}")
print(f"  Recommendation: {result.recommendation}")

# Verify correctness
plain_result = lookup.query_plaintext(drug_a, drug_b)
correct = (result.drug_a == plain_result.drug_a and
           result.severity == plain_result.severity)
print(f"\n  Correctness check: {'PASS' if correct else 'FAIL'}")
print(f"  Total PIR time: {total_ms:.0f} ms "
      f"(query={query_ms:.0f} + server={server_ms:.0f} + decrypt={decrypt_ms:.0f})")

## Cell 5 — FHE: Private Risk Scoring

**Fully Homomorphic Encryption (FHE)** allows computation on encrypted data. The scoring server never sees the patient's features — it computes on ciphertexts and returns an encrypted result that only the agent can decrypt.

**Our model:** A cardiovascular risk scorer using CKKS (approximate arithmetic):
- **Input:** 8 patient features, encrypted as a CKKS vector
- **Computation:** Dot product with weights + polynomial sigmoid activation: $\sigma(x) \approx 0.5 + 0.197x - 0.004x^3$
- **Output:** Risk score in [0, 1], encrypted

The key insight: the polynomial approximation is essential because FHE cannot evaluate arbitrary functions (like true sigmoid). We use a degree-3 polynomial that achieves error < 10⁻⁶ within the clinical input range.

In [ ]:
# ── FHE: Private Risk Scoring ──────────────────────────────────────────
print("=" * 60)
print("  FHE-ENCRYPTED RISK SCORING")
print("=" * 60)

# The scorer was already set up in Cell 3 (with deep params, cv model)
# Let's walk through each step explicitly.

# Step 1: Agent encrypts patient features
print(f"\n[1/3] Agent encrypts patient features...")
print(f"  Plaintext features: {features.to_vector().round(4).tolist()}")

t0 = time.perf_counter()
enc_features = scorer.encrypt_features(features)
encrypt_ms = (time.perf_counter() - t0) * 1000

# Show the encrypted blob
enc_bytes = enc_features.serialize()
print(f"  Encrypted in {encrypt_ms:.0f} ms")
print(f"  Ciphertext size: {len(enc_bytes):,} bytes")
print(f"  First 80 bytes (hex): {enc_bytes[:80].hex()}")
print(f"  >>> Server receives this. It cannot extract any feature values. <<<")

# Step 2: Server computes on encrypted data
print(f"\n[2/3] Server computes risk score on encrypted features...")
print(f"  Model: dot(weights, x) + bias, then cubic sigmoid activation")
print(f"  Weights: {weights.tolist()}")
print(f"  Activation: 0.5 + 0.197*x - 0.004*x^3")

t0 = time.perf_counter()
enc_score = scorer.compute_score_encrypted(enc_features)
compute_ms = (time.perf_counter() - t0) * 1000

score_bytes = enc_score.serialize()
print(f"  Server computation: {compute_ms:.0f} ms")
print(f"  Encrypted result size: {len(score_bytes):,} bytes")
print(f"  SERVER LOG: Computed on encrypted data. Learned: NOTHING.")

# Step 3: Agent decrypts the score
print(f"\n[3/3] Agent decrypts risk score...")
t0 = time.perf_counter()
decrypted_score = scorer.decrypt_score(enc_score)
decrypt_ms = (time.perf_counter() - t0) * 1000
total_fhe_ms = encrypt_ms + compute_ms + decrypt_ms

print(f"  Decrypted in {decrypt_ms:.1f} ms")
print(f"\n  Encrypted score:  {decrypted_score:.6f}")
print(f"  Plaintext score:  {plain_score:.6f}")
error = abs(decrypted_score - plain_score)
print(f"  Absolute error:   {error:.2e}")
print(f"  Within clinical tolerance (0.01): {'YES' if error < 0.01 else 'NO'}")
print(f"\n  Total FHE time: {total_fhe_ms:.0f} ms "
      f"(encrypt={encrypt_ms:.0f} + compute={compute_ms:.0f} + decrypt={decrypt_ms:.1f})")

## Cell 6 — Full Pipeline: PIR + FHE + Local Decision

Now we combine everything into the complete agent pipeline:

1. **PIR lookup** — privately retrieve the drug interaction record
2. **FHE scoring** — privately compute the patient's cardiovascular risk
3. **Local decision** — combine both results on the agent side (no server involved)

We run this for 5 patients, each querying a random drug pair. The tool servers see only encrypted data at every step. The final treatment decision happens entirely on the agent's side.

In [ ]:
# ── Full Pipeline: PIR + FHE + Local Decision ─────────────────────────
print("=" * 60)
print("  FULL ENCRYPTED PIPELINE (5 patients)")
print("=" * 60)

rng = np.random.default_rng(42)
n_pipeline = min(5, len(patients_raw))
severity_map = {"none": 0, "mild": 1, "moderate": 2,
                "severe": 3, "contraindicated": 4}
pipeline_rows = []

for i in range(n_pipeline):
    p = patients_raw[i]
    pf = PatientFeatures(
        age=p["age"], bmi=p["bmi"],
        blood_pressure_systolic=p["systolic_bp"],
        blood_pressure_diastolic=p["diastolic_bp"],
        cholesterol=p["cholesterol_total"],
        glucose=p["glucose_fasting"],
        num_medications=p["num_medications"],
        num_conditions=p["num_conditions"],
    )
    # Pick a random drug pair
    pair_idx = int(rng.integers(0, len(pairs)))
    da, db = pairs[pair_idx]

    # --- PIR phase ---
    t0 = time.perf_counter()
    eq = lookup.create_query(da, db)
    er = lookup.server_process(eq)
    rec = lookup.decrypt_result(er)
    pir_ms = (time.perf_counter() - t0) * 1000

    # --- FHE phase ---
    t0 = time.perf_counter()
    enc_score = scorer.score(pf)
    fhe_ms = (time.perf_counter() - t0) * 1000
    plain_sc = scorer.score_plaintext(pf)

    # --- Local decision ---
    sev_int = severity_map.get(str(rec.severity).lower(),
                               int(rec.severity) if str(rec.severity).isdigit() else 0)
    risk = "low" if enc_score < 0.3 else ("medium" if enc_score < 0.6 else "high")
    if sev_int >= 3:
        decision = "ALERT"
    elif risk == "high" and sev_int >= 2:
        decision = "WARNING"
    else:
        decision = "OK"

    pipeline_rows.append({
        "Patient": p["patient_id"],
        "Drugs": f"{da} + {db}",
        "Severity": rec.severity,
        "Risk Score": f"{enc_score:.4f}",
        "FHE Error": f"{abs(enc_score - plain_sc):.1e}",
        "Decision": decision,
        "PIR (ms)": f"{pir_ms:.0f}",
        "FHE (ms)": f"{fhe_ms:.0f}",
        "Total (ms)": f"{pir_ms + fhe_ms:.0f}",
    })

print(f"\nServer content leakage: 0 bits (by construction)\n")
result_df = pd.DataFrame(pipeline_rows)
result_df

## Cell 7 — Benchmark Results

The full benchmarks (run separately via `make benchmarks` or `python experiments/reproduce_all.py`) test at paper scale: **10,000-record database, 100 patients, 100 runs**. Below we load and display those results if available.

Key metrics reported in the paper:

| Claim | Target |
|-------|--------|
| PIR lookup (10K DB) | ~1,023 ms |
| FHE cubic scoring | ~875 ms |
| Pipeline total | ~1,906 ms |
| Correctness | 100% |
| Max FHE error | < 0.01 |
| Server content leakage | 0 bits |

In [ ]:
# ── Benchmark Results ──────────────────────────────────────────────────
RESULTS = PROJECT_ROOT / "results"
BENCH = RESULTS / "benchmarks"

def _load(path):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        return None

pir_data = _load(BENCH / "pir_benchmarks_scaled.json")
fhe_data = _load(BENCH / "fhe_benchmarks_scaled.json")
pipe_data = _load(BENCH / "pipeline_benchmarks_scaled.json")

if pir_data and fhe_data and pipe_data:
    # ── PIR scaling table ─────────────────────────────────────────
    pir_rows = []
    for r in pir_data["results"]:
        pir_rows.append({
            "DB Size": f"{r['db_size']:,}",
            "Record Size": f"{r['record_size']} B",
            "E2E Latency": f"{r['end_to_end_ms_mean']:.0f} +/- {r['end_to_end_ms_std']:.0f} ms",
            "Server (ms)": f"{r['server_process_ms_mean']:.0f}",
            "Correctness": r["correctness"],
        })
    print("PIR Benchmarks (BFV, 256-byte records):")
    display(pd.DataFrame(pir_rows))

    # ── FHE scoring table ─────────────────────────────────────────
    fhe_rows = []
    for r in fhe_data["results"]:
        fhe_rows.append({
            "Model": r["model"],
            "Preset": r["preset"],
            "n": r["poly_mod"],
            "E2E Latency": f"{r['end_to_end_ms_mean']:.0f} +/- {r['end_to_end_ms_std']:.0f} ms",
            "Max Error": f"{r['max_error']:.1e}",
            "Correctness": r["correctness"],
        })
    print("\nFHE Benchmarks (CKKS):")
    display(pd.DataFrame(fhe_rows))

    # ── Pipeline summary ──────────────────────────────────────────
    bp = max(pipe_data["results"], key=lambda r: r.get("db_size", 0))
    print(f"\nPipeline Summary ({bp['db_size']:,}-record DB, {bp['n_patients']} patients):")
    summary_rows = [
        {"Metric": "PIR Phase", "Value": f"{bp['pir_phase_ms_mean']:.0f} +/- {bp['pir_phase_ms_std']:.0f} ms"},
        {"Metric": "FHE Phase", "Value": f"{bp['fhe_phase_ms_mean']:.0f} +/- {bp['fhe_phase_ms_std']:.0f} ms"},
        {"Metric": "Total Latency", "Value": f"{bp['total_ms_mean']:.0f} +/- {bp['total_ms_std']:.0f} ms"},
        {"Metric": "PIR Correctness", "Value": bp["pir_correctness"]},
        {"Metric": "FHE Correctness", "Value": bp["fhe_correctness"]},
        {"Metric": "Peak Memory", "Value": f"{bp['pipeline_peak_rss_mb']:.0f} MB"},
        {"Metric": "Server Content Leakage", "Value": "0 bits"},
    ]
    display(pd.DataFrame(summary_rows))
else:
    print("Benchmark results not found in results/benchmarks/.")
    print("Run: python experiments/reproduce_all.py --quick")
    print("Or:  make benchmarks")

In [ ]:
# ── Inline Plots ──────────────────────────────────────────────────────
try:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    # --- Plot 1: PIR latency vs DB size ---
    if pir_data:
        sizes = [r["db_size"] for r in pir_data["results"]]
        means = [r["end_to_end_ms_mean"] for r in pir_data["results"]]
        stds  = [r["end_to_end_ms_std"] for r in pir_data["results"]]
        axes[0].errorbar(sizes, means, yerr=stds, fmt="o-",
                         color="#2D6A4F", linewidth=2, capsize=4)
        axes[0].set_xlabel("Database Size (records)")
        axes[0].set_ylabel("Latency (ms)")
        axes[0].set_title("PIR Latency Scaling")
        axes[0].grid(True, alpha=0.3)

    # --- Plot 2: FHE latency by model ---
    if fhe_data:
        models = [f"{r['model']}\n({r['preset']})" for r in fhe_data["results"]]
        fhe_means = [r["end_to_end_ms_mean"] for r in fhe_data["results"]]
        fhe_stds  = [r["end_to_end_ms_std"] for r in fhe_data["results"]]
        colors = ["#1B4332" if "deep" in r["preset"] else "#52B788"
                  for r in fhe_data["results"]]
        bars = axes[1].bar(models, fhe_means, yerr=fhe_stds,
                           color=colors, capsize=4, width=0.5)
        axes[1].bar_label(bars, fmt="%.0f ms", fontsize=9)
        axes[1].set_ylabel("Latency (ms)")
        axes[1].set_title("FHE Scoring Latency")
        axes[1].grid(True, alpha=0.3, axis="y")

    # --- Plot 3: Pipeline breakdown ---
    if pipe_data:
        bp = max(pipe_data["results"], key=lambda r: r.get("db_size", 0))
        phases = ["PIR", "FHE", "Decision"]
        phase_ms = [bp["pir_phase_ms_mean"], bp["fhe_phase_ms_mean"],
                    bp["decision_ms_mean"]]
        phase_colors = ["#2D6A4F", "#40916C", "#B7E4C7"]
        bars = axes[2].bar(phases, phase_ms, color=phase_colors, width=0.5)
        axes[2].bar_label(bars, fmt="%.0f ms", fontsize=9)
        axes[2].set_ylabel("Latency (ms)")
        axes[2].set_title(f"Pipeline Breakdown ({bp['db_size']:,} DB)")
        axes[2].grid(True, alpha=0.3, axis="y")

    fig.tight_layout()
    plt.show()
except Exception as e:
    print(f"Could not generate plots: {e}")

## Cell 8 — Feasibility Boundary

Not all tool calls can be encrypted. Our paper identifies a **feasibility boundary** that separates what FHE/PIR can protect from what remains infeasible:

| Category | Primitive | Status |
|----------|-----------|--------|
| **Structured retrieval** (key-value lookup) | PIR | Feasible (~1s) |
| **Bounded polynomial computation** (linear models, low-degree activations) | FHE/CKKS | Feasible (~0.9s) |
| **Access-pattern hiding** (which records, how often) | ORAM | Theoretical (high overhead) |
| **Arbitrary code execution** (branching, loops, I/O) | None | Infeasible |

The figure below visualizes this boundary. The shaded green region represents tool calls that our approach can protect today. The slider (if `ipywidgets` is available) lets you adjust the latency budget to see how the feasibility region changes.

In [ ]:
# ── Feasibility Boundary ──────────────────────────────────────────────

def plot_feasibility_boundary(latency_budget_ms=2000):
    """Plot the feasibility boundary with an adjustable latency budget."""
    fig, ax = plt.subplots(figsize=(10, 6))

    # Define tool-call categories along a "computation complexity" axis
    categories = [
        {"name": "Key-Value\nLookup", "complexity": 1, "latency": 1023,
         "primitive": "PIR", "feasible": True},
        {"name": "Linear\nScoring", "complexity": 2, "latency": 193,
         "primitive": "FHE (CKKS)", "feasible": True},
        {"name": "Polynomial\nScoring", "complexity": 3, "latency": 875,
         "primitive": "FHE (CKKS)", "feasible": True},
        {"name": "Full\nPipeline", "complexity": 4, "latency": 1906,
         "primitive": "PIR + FHE", "feasible": True},
        {"name": "Access\nPatterns", "complexity": 5.5, "latency": 5000,
         "primitive": "ORAM", "feasible": False},
        {"name": "Arbitrary\nExecution", "complexity": 7, "latency": None,
         "primitive": "None", "feasible": False},
    ]

    # Shaded feasibility region
    ax.axhspan(0, latency_budget_ms, alpha=0.08, color="green",
               label=f"Feasibility region (< {latency_budget_ms} ms)")
    ax.axhline(y=latency_budget_ms, color="green", linestyle="--",
               linewidth=2, alpha=0.6, label=f"Budget: {latency_budget_ms} ms")

    # Plot each category
    for cat in categories:
        x = cat["complexity"]
        if cat["latency"] is not None and cat["feasible"]:
            color = "#2D6A4F" if cat["latency"] <= latency_budget_ms else "#E63946"
            marker = "o"
            ax.plot(x, cat["latency"], marker, markersize=14, color=color,
                    zorder=5)
            ax.annotate(f'{cat["latency"]:,} ms\n({cat["primitive"]})',
                        (x, cat["latency"]), textcoords="offset points",
                        xytext=(0, 18), ha="center", fontsize=9,
                        fontweight="bold", color=color)
        elif cat["latency"] is not None:
            ax.plot(x, cat["latency"], "s", markersize=14, color="#E9C46A",
                    zorder=5)
            ax.annotate(f'~{cat["latency"]:,} ms\n({cat["primitive"]})',
                        (x, cat["latency"]), textcoords="offset points",
                        xytext=(0, 18), ha="center", fontsize=9, color="#B8860B")
        else:
            ax.annotate("INFEASIBLE", (x, latency_budget_ms * 1.8),
                        ha="center", fontsize=11, fontweight="bold",
                        color="#E63946")
            ax.plot(x, latency_budget_ms * 1.8, "X", markersize=16,
                    color="#E63946", zorder=5)

    x_labels = [c["name"] for c in categories]
    x_pos = [c["complexity"] for c in categories]
    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels, fontsize=10)
    ax.set_ylabel("Latency (ms)", fontsize=12)
    ax.set_xlabel("Computation Complexity", fontsize=12)
    ax.set_title("FHE/PIR Feasibility Boundary for AI Agent Tool Calls",
                 fontsize=14, fontweight="bold")
    ax.set_ylim(0, max(latency_budget_ms * 2.2, 5500))
    ax.legend(loc="upper left", fontsize=10)
    ax.grid(True, alpha=0.2)

    fig.tight_layout()
    plt.show()

# Try interactive slider; fall back to static plot
try:
    from ipywidgets import interact, IntSlider
    interact(
        plot_feasibility_boundary,
        latency_budget_ms=IntSlider(
            value=2000, min=500, max=10000, step=250,
            description="Budget (ms):", style={"description_width": "initial"},
            layout={"width": "400px"},
        ),
    )
except ImportError:
    print("(ipywidgets not available -- showing static plot with 2000 ms budget)")
    plot_feasibility_boundary(2000)